# NPZ Inspector

fair_eval 파이프라인에서 생성되는 `.npz` 파일 들여다보기 + 원하는 데이터만 뽑기.

| 파일 | 위치 | 내용 |
|------|------|------|
| `data.npz` | `EVAL_DIR/` | X_all, y_all (전체 데이터셋) |
| `splits.npz` | `EVAL_DIR/` | te_fixed, oracle_idx, tr_i, val_i |
| `class_data.npz` | `ensembles/split_{i}/c{c}/` | X_class (해당 split 의 class train) |
| `surrogate_data.npz` | `ensembles/.../ebm_{k}/` | X_ebm + y_ebm (real + 4 surrogate) |
| `negatives.npz` | `ensembles/.../ebm_{k}/` | X_neg (4개 corner) |
| `negatives_union.npz` | `ensembles/.../c{c}/` | K 멤버의 negative 합집합 |
| `heatmap_pad*.npz` | `ensembles/.../c{c}/` | PCA grid energy heatmap |
| `*_traj.npz` | `samples/split_{i}/{cfg}/` | VP-SGLD step 별 위치 |

## 0. Setup

In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd

os.chdir('/home/work/JooKyung/TabEBM')
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)
print('ready')

## 1. 탐색 유틸

- `summary(path)` — npz 의 모든 key 를 **DataFrame 테이블**로 반환 (shape/dtype/size/min/max/sample)
- `peek(path)` — 같은 내용 print
- `list_eval_dirs()` — 사용 가능한 EVAL_DIR 목록

In [ ]:
def summary(path):
    """npz key 들을 DataFrame 테이블로 반환."""
    path = Path(path)
    rows = []
    with np.load(path, allow_pickle=True) as z:
        for k in z.files:
            v = z[k]
            is_num = np.issubdtype(v.dtype, np.number) if v.size else False
            rows.append({
                'key':    k,
                'shape':  str(v.shape) if v.ndim else '()',
                'dtype':  str(v.dtype),
                'size':   int(v.size),
                'min':    float(v.min()) if is_num and v.size else None,
                'max':    float(v.max()) if is_num and v.size else None,
                'sample': np.array_str(v.ravel()[:3], precision=3) if v.size else '',
            })
    df = pd.DataFrame(rows)
    print(f'{path}  ({path.stat().st_size/1024:.1f} KB, {len(df)} keys)')
    return df

def peek(path):
    return summary(path)

def list_eval_dirs(root='experiments/fair_eval'):
    p = Path(root)
    return sorted([d for d in p.iterdir() if d.is_dir() and (d / 'config.json').exists()]) if p.exists() else []

for d in list_eval_dirs():
    cfg = json.loads((d / 'config.json').read_text())
    print(f'  {d.name:<55}  dataset={cfg.get("dataset"):<10} n_real={cfg.get("n_real"):<5} K={cfg.get("K")}')

## 2. EVAL_DIR 선택

In [ ]:
fair_root = Path('experiments/fair_eval')
_candidates = sorted([
    p for p in fair_root.iterdir()
    if p.is_dir() and (p / 'config.json').exists()
       and (p / 'ensembles' / 'split_0' / 'c0' / 'meta.json').exists()
])
EVAL_DIR = _candidates[-1] if _candidates else Path('experiments/fair_eval/NONE')
# EVAL_DIR = Path('experiments/fair_eval/20260417_215937')   # <-- legacy stock (재현 불가)
assert (EVAL_DIR / 'config.json').exists(), f'없음: {EVAL_DIR}'
print(f'EVAL_DIR: {EVAL_DIR}')

assert EVAL_DIR.exists(), f'없음: {EVAL_DIR}'
config = json.loads((EVAL_DIR / 'config.json').read_text())
print(json.dumps(config, indent=2, ensure_ascii=False))


## 3. `data.npz` — 전체 X / y

- `X_all`: (N, D) float — 전체 데이터
- `y_all`: (N,) int   — 0-indexed label

In [ ]:
summary(EVAL_DIR / 'data.npz')

In [ ]:
# === 원하는 데이터만 뽑기 ===
z = np.load(EVAL_DIR / 'data.npz')
X_all, y_all = z['X_all'], z['y_all']

# (1) 전체 출력
print(f'X_all.shape = {X_all.shape},  y_all bincount = {np.bincount(y_all)}')

# (2) 처음 N개만
print(f'X_all[:3] =\n{X_all[:3]}')

# (3) 특정 class 만
X_class0 = X_all[y_all == 0]
print(f'class 0 샘플 수: {len(X_class0)}')

# (4) 특정 feature (column) 만
X_col0 = X_all[:, 0]                 # 첫 번째 feature
X_cols = X_all[:, [0, 1, 5]]         # 여러 feature
print(f'첫 feature 평균 = {X_col0.mean():.3f},  std = {X_col0.std():.3f}')

# (5) 조건 필터 (boolean indexing)
mask = X_all[:, 0] > 0                # 첫 feature > 0 인 샘플
X_pos = X_all[mask]
print(f'첫 feature > 0 인 샘플: {mask.sum()}개')

# (6) 특정 인덱스들만
idx = np.array([0, 5, 10, 20])
X_sel = X_all[idx]

# (7) 통계 요약 DataFrame
stats = pd.DataFrame({
    'feature': [f'f{i}' for i in range(X_all.shape[1])],
    'mean':    X_all.mean(0),
    'std':     X_all.std(0),
    'min':     X_all.min(0),
    'max':     X_all.max(0),
})
stats

## 4. `splits.npz` — train / val / test 인덱스

paper Figure 9 protocol 키:
- `te_fixed` — 전 split 공통 test (min(N/2, 500))
- `oracle_idx` — te_fixed 와 disjoint 한 pool
- `tr_i`, `val_i` (i=0..n_splits-1) — split i 의 train / val (oracle 부분집합, 80/20)

In [ ]:
summary(EVAL_DIR / 'splits.npz')

In [ ]:
# === 원하는 데이터만 뽑기 ===
sp = np.load(EVAL_DIR / 'splits.npz')

# (1) fixed test / oracle
te_fixed = sp['te_fixed']                 # 모든 split 공통
oracle   = sp['oracle_idx']

# (2) 특정 split 의 train / val
split_i = 0
tr   = sp[f'tr_{split_i}']
val  = sp[f'val_{split_i}']

# (3) 실제 데이터로 변환 — splits 는 인덱스이므로 X_all 과 함께
X_tr, y_tr = X_all[tr],       y_all[tr]
X_val, y_val = X_all[val],    y_all[val]
X_te, y_te = X_all[te_fixed], y_all[te_fixed]

# (4) disjoint 검증
print(f'tr ∩ val: {np.intersect1d(tr, val).size}')
print(f'tr ∩ te_fixed: {np.intersect1d(tr, te_fixed).size}')
print(f'val ∩ te_fixed: {np.intersect1d(val, te_fixed).size}')
print(f'tr ⊆ oracle: {np.all(np.isin(tr, oracle))}')

# (5) 모든 split 을 dict 로 한 번에
all_splits = {i: (sp[f'tr_{i}'], sp[f'val_{i}']) for i in range(config['n_splits'])}

# (6) split 간 train overlap 살펴보기 (oracle 에서 sampling 하므로 자연스럽게 겹침)
overlap = np.zeros((config['n_splits'], config['n_splits']), dtype=int)
for i in range(config['n_splits']):
    for j in range(config['n_splits']):
        overlap[i, j] = np.intersect1d(sp[f'tr_{i}'], sp[f'tr_{j}']).size
print(f'\nsplit 간 train overlap (대각=본인, 대각외=공통 샘플 수):')
print(pd.DataFrame(overlap, index=[f's{i}' for i in range(config['n_splits'])],
                           columns=[f's{i}' for i in range(config['n_splits'])]))

# (3) per-split preprocessor (paper B.3 — tr-only imputer + scaler)
if f'imp_mean_{split_i}' in sp.files:
    imp = sp[f'imp_mean_{split_i}']
    sm  = sp[f'scl_mean_{split_i}']
    ss  = sp[f'scl_std_{split_i}']
    print(f'\n--- split {split_i} preprocessor (D={len(imp)}) ---')
    print(f'  imp_mean[:5] = {imp[:5].round(3)}')
    print(f'  scl_mean[:5] = {sm[:5].round(3)}')
    print(f'  scl_std[:5]  = {ss[:5].round(3)}')
else:
    print(f'\nlegacy splits.npz — preprocessor params 없음 (01 재실행 필요)')


## 5. `class_data.npz` — 특정 split × class 의 train 샘플

In [ ]:
SPLIT_I = 0
CLASS_C = 0
class_dir = EVAL_DIR / 'ensembles' / f'split_{SPLIT_I}' / f'c{CLASS_C}'
summary(class_dir / 'class_data.npz')

In [ ]:
# === 원하는 데이터만 뽑기 ===
cd = np.load(class_dir / 'class_data.npz')
X_class = cd['X_class']              # (n_pos, D)

# (1) meta.json 도 함께
meta = json.loads((class_dir / 'meta.json').read_text())
print(f'X_class.shape = {X_class.shape}')
print(f'meta: {meta}')

# (2) 통계
print(f'\nmean per feature: {X_class.mean(0)[:5]} ...')   # 처음 5 feature
print(f'std per feature:  {X_class.std(0)[:5]} ...')

# (3) PCA 로 2D 투영 (heatmap 같이 쓰려면)
from sklearn.decomposition import PCA
Z = PCA(n_components=2).fit_transform(X_class)
print(f'\nPCA 2D: {Z.shape}, range x={Z[:,0].min():.2f}..{Z[:,0].max():.2f}')

## 6. EBM member — `surrogate_data.npz` + `negatives.npz`

한 K-th 멤버 디렉터리에 들어있는 파일들.

In [ ]:
K_IDX = 0
ebm_dir = class_dir / f'ebm_{K_IDX}'

print('=== surrogate_data ===')
display(summary(ebm_dir / 'surrogate_data.npz'))
print('=== negatives ===')
display(summary(ebm_dir / 'negatives.npz'))

In [ ]:
# === 원하는 데이터만 뽑기 ===
sd = np.load(ebm_dir / 'surrogate_data.npz')
ng = np.load(ebm_dir / 'negatives.npz')

# (1) X_ebm = [real positives; 4 negatives] 이 붙어있음
X_ebm, y_ebm = sd['X_ebm'], sd['y_ebm']
n_pos = int((y_ebm == 0).sum())
n_neg = int((y_ebm == 1).sum())
print(f'X_ebm {X_ebm.shape}: positives {n_pos}, negatives {n_neg}')

# (2) real / surrogate 분리
X_real = X_ebm[y_ebm == 0]
X_surrogate = X_ebm[y_ebm == 1]
print(f'real:      {X_real.shape}')
print(f'surrogate: {X_surrogate.shape}')

# (3) negatives.npz 로 더 직접 접근
X_neg = ng['X_neg']                       # (4, D)
print(f'\nX_neg (4 corners):\n{X_neg}')
# antipodal pair 확인
print(f'\nantipodal 확인 (0 + 1, 2 + 3 이 거의 0 인가):')
print(f'  X_neg[0] + X_neg[1] = {X_neg[0] + X_neg[1]}')
print(f'  X_neg[2] + X_neg[3] = {X_neg[2] + X_neg[3]}')

## 7. `negatives_union.npz` — K 멤버 전체 negatives 합집합

In [ ]:
summary(class_dir / 'negatives_union.npz')

In [ ]:
# === 원하는 데이터만 뽑기 ===
u = np.load(class_dir / 'negatives_union.npz')
X_neg_all   = u['X_neg']               # (4*K, D) 전체
member_idx  = u['member_idx']          # 각 row 가 어느 멤버에서 왔는지
alpha_per   = u['alpha_per_member']    # 멤버별 distance
K           = int(u['K'])

print(f'K={K}, 총 negatives = {len(X_neg_all)}')

# (1) 특정 멤버의 negatives 만
for k in range(min(K, 3)):
    mask = member_idx == k
    print(f'  멤버 {k}: {mask.sum()}개  (alpha={alpha_per[k]})')

# (2) 중복 제거 후 unique negative 개수
X_unique = np.unique(X_neg_all, axis=0)
print(f'\nunique negatives: {len(X_unique)} / {len(X_neg_all)}  (중복 없으면 같음)')

## 8. `heatmap_pad*.npz` — PCA grid energy cache

01 의 heatmap 시각화에서 만들어지는 캐시.

In [ ]:
hm_files = sorted(class_dir.glob('heatmap_*.npz'))
print(f'heatmap 캐시: {len(hm_files)}개')
for f in hm_files:
    print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB)')

if hm_files:
    summary(hm_files[0])

In [ ]:
# === 원하는 데이터만 뽑기 ===
if hm_files:
    hm = np.load(hm_files[0])
    K = int(hm['K'])
    ZZ1, ZZ2 = hm['ZZ1'], hm['ZZ2']
    Z_class  = hm['Z_class']             # real pos 의 PCA 투영
    var_ratio = float(hm['var_ratio'])

    # (1) 한 멤버만
    E_0 = hm['E_0']

    # (2) 전 멤버 stack → mean/std
    E_stack = np.stack([hm[f'E_{k}'] for k in range(K)])
    E_mean  = E_stack.mean(0)
    E_std   = E_stack.std(0, ddof=1)

    print(f'grid {ZZ1.shape}, PCA var 설명 {var_ratio:.1%}')
    print(f'E_mean range: {E_mean.min():.3f} .. {E_mean.max():.3f}')
    print(f'E_std  range: {E_std.min():.3f} .. {E_std.max():.3f}')

    # (3) 특정 grid 영역만 (예: 왼쪽 절반)
    h, w = E_mean.shape
    E_left = E_mean[:, :w//2]
    print(f'왼쪽 절반 mean: {E_left.mean():.3f}')

## 9. `*_traj.npz` — VP-SGLD 궤적

02 에서 `TRAJ_CHECKPOINTS` 켰을 때만 생성.

In [ ]:
traj_files = list((EVAL_DIR / 'samples').rglob('*_traj.npz'))
print(f'trajectory 파일: {len(traj_files)}개')
for f in traj_files[:5]:
    print(f'  {f.relative_to(EVAL_DIR)}')

if traj_files:
    summary(traj_files[0])

In [ ]:
# === 원하는 데이터만 뽑기 ===
if traj_files:
    tj = np.load(traj_files[0])
    steps = tj['steps']                   # checkpointed step 번호 리스트
    print(f'steps = {steps.tolist()}')

    # (1) 마지막 step
    final_step = int(steps[-1])
    X_final = tj[f'step_{final_step}']    # (B, D) float16
    print(f'final (step {final_step}) shape = {X_final.shape}, dtype={X_final.dtype}')

    # (2) 전체 trajectory 를 (T, B, D) stack 으로
    X_traj = np.stack([tj[f'step_{s}'] for s in steps])
    print(f'stacked: {X_traj.shape}')

    # (3) 특정 chain (B 축) 만
    chain_0 = X_traj[:, 0, :]              # chain 0 의 전 step
    print(f'chain 0 trajectory: {chain_0.shape}')

    # (4) step 별 평균 이동 거리 (수렴 판단)
    dists = np.linalg.norm(np.diff(X_traj, axis=0), axis=2).mean(axis=1)
    print(f'step 별 평균 이동거리 (수렴 체크):')
    for s_from, s_to, d in zip(steps[:-1], steps[1:], dists):
        print(f'  step {s_from:>4d} → {s_to:>4d}: {d:.4f}')

## 10. `samples/split_{i}/{cfg}/c{c}.npy` — 생성된 synthetic

이건 `.npz` 가 아니라 `.npy` (단일 배열).

In [ ]:
samples_dir = EVAL_DIR / 'samples'
if samples_dir.exists():
    # 한 split 의 모든 config
    split0 = samples_dir / 'split_0'
    cfgs = [p.name for p in split0.iterdir() if p.is_dir()]
    print(f'split 0 configs: {len(cfgs)}개')
    for cfg in cfgs[:5]:
        for c in config['classes']:
            f = split0 / cfg / f'c{c}.npy'
            if f.exists():
                arr = np.load(f, mmap_mode='r')   # mmap 으로 메모리 절약
                print(f'  {cfg}/c{c}.npy  shape={arr.shape}  dtype={arr.dtype}')

    # === 원하는 샘플만 뽑기 ===
    if cfgs:
        cfg_name = cfgs[0]
        X_syn_c0 = np.load(split0 / cfg_name / 'c0.npy')
        X_syn_c1 = np.load(split0 / cfg_name / 'c1.npy')
        print(f'\n{cfg_name} — class 0: {X_syn_c0.shape}, class 1: {X_syn_c1.shape}')

        # 처음 100 개만
        X_sub = X_syn_c0[:100]
        # 특정 인덱스만
        X_pick = X_syn_c0[[0, 10, 20]]

## 11. 치트시트 — npz 데이터 뽑기 패턴 정리

```python
# ─── 기본 ─────────────────────────────────────
z = np.load('x.npz')
print(z.files)                  # 모든 key 목록
x = z['key_name']               # 특정 key (lazy — 필요한 것만 메모리)

# ─── with 문 (자동 close) ────────────────────
with np.load('x.npz') as z:
    tr = {i: z[f'tr_{i}'] for i in range(10)}

# ─── 전체 dict 로 ────────────────────────────
all_data = dict(np.load('x.npz'))

# ─── slicing / fancy indexing ────────────────
X = z['X_all']
X[:100]                         # 처음 100
X[:, [0, 3, 5]]                 # 특정 feature
X[X[:, 0] > 0]                  # boolean mask
X[[1, 5, 10]]                   # 특정 row

# ─── 메모리 큰 경우 mmap ─────────────────────
arr = np.load('big.npy', mmap_mode='r')   # .npy 만 가능
part = arr[100:200]                        # 필요한 슬라이스만 디스크에서 읽음

# ─── 재저장 (subset) ─────────────────────────
np.savez('subset.npz', tr_0=tr_0, val_0=val_0)
np.savez_compressed('small.npz', big=arr)  # 압축 (느림, 공간 절약)

# ─── pickle 된 dict/list 꺼내기 ───────────────
z = np.load('x.npz', allow_pickle=True)
obj = z['dict_field'].item()    # 0-d object array → dict/list

# ─── 파일 크기 / key 수 확인 ─────────────────
from pathlib import Path
p = Path('x.npz'); print(p.stat().st_size / 1024, 'KB')
with np.load(p) as z: print(len(z.files), 'keys')
```